> ⚠️ **作業中 (Work in Progress)**:このノートブックは現在開発中です。一部のコードが不完全であったり変更される可能性があります。

## 📋 目次

- [評価概要](#評価概要)
- [環境設定](#環境-設定)
- [評価基準の理解](#評価基準の理解)
- [評価の実行と結果分析](#評価の実行と結果分析)
- [評価のベストプラクティス](#評価のベストプラクティス)

## 🎯 学習目標

- AIエージェント評価の重要性の理解
- Foundryの自動評価機能の活用
- 様々な評価指標の意味と活用方法の学習
- 合成データを使用した評価の実行
- 評価結果の解釈と改善方法の導出

## ⏱️ 予想所要時間

約10分

## 評価概要

### なぜ評価が重要なのか?

AI エージェントを本番環境にデプロイすること前に次の事項を検証する必要がありします:

```
正確性 → 関連性 → 一貫性 → 自然さ → 安全性
```

評価ないがデプロイすると:
- ❌ 不正確な回答でユーザー 信頼低下
- ❌ 関連ないはレスポンスでユーザー 経験悪化
- ❌ 一貫性ないは品質でブランドが未知損傷
- ❌ 不適切なコンテンツ作成で法的問題

### Microsoft Foundryの評価機能

Foundryは次のを自動化します:
- ✅ テストデータ作成 (Synthetic generation)
- ✅ 様々な評価指標適用
- ✅ 大規模評価実行
- ✅ 結果可視化および分析

## 環境設定

評価をための環境を設定します.

In [ ]:
# 環境変数ロード
import json
import os
import subprocess

# PATH 環境変数の設定 (Azure CLIを見つけられるように)
possible_paths = [
  "/opt/homebrew/bin", # macOS (Apple Silicon)
  "/usr/local/bin",   # macOS (Intel) / Linux
  "/usr/bin",      # Linux / GitHub Codespaces
  "/home/linuxbrew/.linuxbrew/bin" # Linux Homebrew
]

az_path = None
try:
  result = subprocess.run(['which', 'az'], capture_output=True, text=True)
  if result.returncode == 0:
    az_path = os.path.dirname(result.stdout.strip())
except:
  pass

paths_to_add = []
if az_path and az_path not in os.environ.get("PATH", ""):
  paths_to_add.append(az_path)
else:
  for path in possible_paths:
    if os.path.exists(path) and path not in os.environ.get("PATH", ""):
      paths_to_add.append(path)

if paths_to_add:
  new_path = ":".join(paths_to_add) + ":" + os.environ.get("PATH", "")
  os.environ["PATH"] = new_path

# が前ノートブックで保存した設定ファイルのロード
config_file = ".foundry_config.json"
try:
  with open(config_file, 'r') as f:
    config = json.load(f)
  
  # 環境変数設定
  FOUNDRY_NAME = config.get("FOUNDRY_NAME")
  RESOURCE_GROUP = config.get("RESOURCE_GROUP")
  LOCATION = config.get("LOCATION")
  TENANT_ID = config.get("TENANT_ID")
  PROJECT_NAME = config.get("PROJECT_NAME", "proj-default")
  PROJECT_ENDPOINT = config.get("FOUNDRY_ENDPOINT")
  
  # 環境変数でも設定 (他のツールが使用できるように)
  os.environ["FOUNDRY_NAME"] = FOUNDRY_NAME
  os.environ["LOCATION"] = LOCATION
  os.environ["RESOURCE_GROUP"] = RESOURCE_GROUP
  os.environ["AZURE_SUBSCRIPTION_ID"] = config.get("AZURE_SUBSCRIPTION_ID", "")
  os.environ["_ENDPOINT"] = PROJECT_ENDPOINT
  
  print(f"✅ 設定ファイル '{config_file}'で環境変数をロードしました.")
  print(f"\n📌 Foundry Name:{FOUNDRY_NAME}")
  print(f"📌 Resource Group:{RESOURCE_GROUP}")
  print(f"📌 Location:{LOCATION}")
  print(f"📌 プロジェクトエンドポイント:{PROJECT_ENDPOINT}")
  
except FileNotFoundError:
  print(f"⚠️ '{config_file}' ファイルを見つかりません.")
  print("💡 01-setup.ipynbを先に実行して環境を設定してください.")
  raise

# 必須パッケージインストール
%pip install -q azure-ai-evaluation azure-ai-projects azure-identity

from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential

print(f"\n💡 使用するプロジェクトエンドポイント:{PROJECT_ENDPOINT}")


In [ ]:
# 簡単なエージェント評価例
from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential

credential = DefaultAzureCredential()
project_client = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=credential)

# テストデータの準備
test_queries = [
  "Pythonでリストをソートする方法は?",
  "機械学習とディープラーニングの違いを説明してください.",
  "ソウルの人口は何人ですか?",
  "最近のAI技術の動向を教えてください.",
  "クラウドコンピューティングのメリットは何ですか?"
]

print("📊 エージェントテスト開始...\n")
print("=" * 80)

# 自動で ModelRouterAgent 検索
try:
  agents = list(project_client.agents.list())
  agent_router = next((a for a in agents if a.name == "ModelRouterAgent"), None)
  
  if not agent_router:
    print("⚠️ 'ModelRouterAgent'を見つかりません.")
    print("💡 03-agents.ipynbを先に実行してエージェントを作成してください.")
    raise ValueError("Agent not found")
  
  print(f"✅ 使用するエージェント:{agent_router.name} (ID:{agent_router.id})\n")
  
  # デモ用 Mock レスポンス (実際環境では Portal 評価推奨)
  print("\n💡 参考:SDK バージョン制限でにより Mock レスポンスを使用します.")
  print("  実際大規模評価は Azure Portalを使用してください.\n")
  
  mock_responses = [
    "Pythonでリストをソートするには `sort()` メソッドや `sorted()` 関数を使用するできるあります. `list.sort()`はリストをその場でソートして, `sorted(list)`は新しいで運ソートされたリストを返却します.",
    "機械学習は明示的プログラミングないがデータでパターンを学習するは技術がであり, ディープラーニングは人工ニューラルネットワークを使用するは機械学習のした分野です. ディープラーニングはより複雑なパターンを学習できるできるありだけより多いはデータとコンピューティングパワーが必要です.",
    "ソウルの人口は約 950だけ名です. (2023年基準)",
    "最近 AI 技術動向では大規模言語モデル(LLM)の発展, マルチモーダル AI, 作成型 AIの拡散, AI エージェントシステムの発展などがあります.",
    "クラウドコンピューティングの主要メリットは拡張性, コスト効率性, アクセス性, 自動更新, 災害復旧機能などです. 必要にに従ってリソースを柔軟に調整するできるあって効率的です."
  ]
  
  responses = []
  for i, query in enumerate(test_queries, 1):
    print(f"\n[質問 {i}/{len(test_queries)}]:{query}")
    response = mock_responses[i-1]
    responses.append({"query":query, "response":response})
    print(f"[レスポンス]:{response[:100]}...")
  
  print("\n" + "=" * 80)
  print(f"\n✅ {len(test_queries)}個の質問テスト完了!")
  print(f"\n💡 ポータルでより詳細な評価を行ってください:")
  print("  https://ai.azure.com > Build > Evaluations")

except Exception as e:
  print(f"\n⚠️ エラー 発生:{e}")
  print("\n💡 解決方法:")
  print("  1. 03-agents.ipynbをまず実行して ModelRouterAgentを作成")
  print("  2. Azureにログインしているか確認 (az login)")
  print("  3. FOUNDRY_NAMEが正しいか確認")

### Azure AI Evaluation SDK 使用すること

SDKを通じてプログラミング方式で評価を実行するできるあります. 様々な Evaluatorを使用してエージェントレスポンスの品質を測定します.

In [ ]:
# Azure AI Evaluation SDK Evaluator インポート
from azure.ai.evaluation import (
  CoherenceEvaluator,
  FluencyEvaluator,
  GroundednessEvaluator,
  RelevanceEvaluator,
)

# Azure OpenAI モデル設定 (評価用)
model_config = {
  "azure_endpoint":f"https://{FOUNDRY_NAME}.openai.azure.com/",
  "api_version":"2024-08-01-preview",
  "azure_deployment":"gpt-5.1", # 評価に使用するモデル (するがオープン含む)
}

# Evaluator インスタンス作成
coherence_evaluator = CoherenceEvaluator(model_config=model_config)
fluency_evaluator = FluencyEvaluator(model_config=model_config)
groundedness_evaluator = GroundednessEvaluator(model_config=model_config)
relevance_evaluator = RelevanceEvaluator(model_config=model_config)

print("✅ Evaluator インスタンスが作成なりました:")
print("  - CoherenceEvaluator:論理的一貫性評価")
print("  - FluencyEvaluator:自然さ評価")
print("  - GroundednessEvaluator:事実ベース評価")
print("  - RelevanceEvaluator:関連性評価")

In [ ]:
# 評価実行例
# 上で収集した responses データを使用して評価します

print("📊 SDK ベース評価実行中...\n")
print("=" * 80)

evaluation_results = []

for i, item in enumerate(responses, 1):
  query = item["query"]
  response = item["response"]
  
  # 評価用コンテキスト (実際環境では RAGでが取得したドキュメント使用)
  context = "一般的なナレッジベース質問にに対するレスポンスです."
  
  print(f"\n[サンプル {i}/{len(responses)}]")
  print(f"質問:{query[:50]}...")
  
  # 各 Evaluatorで評価
  coherence_score = coherence_evaluator(query=query, response=response)
  fluency_score = fluency_evaluator(query=query, response=response)
  groundedness_score = groundedness_evaluator(query=query, response=response, context=context)
  relevance_score = relevance_evaluator(query=query, response=response, context=context)
  
  result = {
    "query":query,
    "response":response[:100],
    "coherence":coherence_score.get("coherence", "N/A"),
    "fluency":fluency_score.get("fluency", "N/A"),
    "groundedness":groundedness_score.get("groundedness", "N/A"),
    "relevance":relevance_score.get("relevance", "N/A"),
  }
  evaluation_results.append(result)
  
  print(f" Coherence:{result['coherence']}/5")
  print(f" Fluency:{result['fluency']}/5")
  print(f" Groundedness:{result['groundedness']}/5")
  print(f" Relevance:{result['relevance']}/5")

print("\n" + "=" * 80)
print(f"\n✅ {len(responses)}個サンプル評価完了!")

In [ ]:
# 評価結果概要
import statistics

print("=" * 80)
print("📈 評価結果概要")
print("=" * 80)

# 各指標別平均計算
metrics = ["coherence", "fluency", "groundedness", "relevance"]
averages = {}

for metric in metrics:
  scores = [r[metric] for r in evaluation_results if isinstance(r[metric], (int, float))]
  if scores:
    averages[metric] = statistics.mean(scores)
  else:
    averages[metric] = "N/A"

print(f"\n📊 Overall Scores:")
print(f"  Coherence:  {averages['coherence']:.2f}/5.0" if isinstance(averages['coherence'], float) else f"  Coherence:  {averages['coherence']}")
print(f"  Fluency:   {averages['fluency']:.2f}/5.0" if isinstance(averages['fluency'], float) else f"  Fluency:   {averages['fluency']}")
print(f"  Groundedness:{averages['groundedness']:.2f}/5.0" if isinstance(averages['groundedness'], float) else f"  Groundedness:{averages['groundedness']}")
print(f"  Relevance:  {averages['relevance']:.2f}/5.0" if isinstance(averages['relevance'], float) else f"  Relevance:  {averages['relevance']}")

# Pass/Fail 基準 (4.0 が上が場合 Pass)
threshold = 3.5
passed = sum(1 for r in evaluation_results 
       if all(isinstance(r[m], (int, float)) and r[m] >= threshold for m in metrics))
pass_rate = (passed / len(evaluation_results)) * 100 if evaluation_results else 0

print(f"\n✅ Pass Rate:{pass_rate:.1f}% ({passed}/{len(evaluation_results)} samples)")
print(f"  (基準:すべての指標 ≥ {threshold})")

print("\n" + "=" * 80)
print("\n💡 推奨事項:")
if averages.get('groundedness', 5) < 4.0:
  print("  ⚠️ Groundedness 改善必要:Knowledge Base 補強または Instructions 修正")
if averages.get('relevance', 5) < 4.0:
  print("  ⚠️ Relevance 改善必要:エージェント Instructionsをより明確に")
if averages.get('coherence', 5) < 4.0:
  print("  ⚠️ Coherence 改善必要:レスポンス構造化ガイドライン追加")
if averages.get('fluency', 5) < 4.0:
  print("  ⚠️ Fluency 改善必要:モデルアップグレがするまたはプロンプト改善")
if pass_rate >= 80:
  print("  ✅ 全体的で良好なパフォーマンスです!")

# 結果可視化 (選択事項)
try:
  import matplotlib.pyplot as plt
  
  # 指標別平均スコア棒 グラフ
  valid_metrics = {k:v for k, v in averages.items() if isinstance(v, (int, float))}
  
  if valid_metrics:
    plt.figure(figsize=(10, 6))
    plt.bar(valid_metrics.keys(), valid_metrics.values())
    plt.axhline(y=threshold, color='r', linestyle='--', label=f'Threshold ({threshold})')
    plt.ylim(0, 5)
    plt.ylabel('Score')
    plt.title('Evaluation Results Summary')
    plt.legend()
    plt.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    print("\n📊 可視化完了!")
except ImportError:
  print("\n💡 可視化をために matplotlib インストール:pip install matplotlib")

### Foundry 提供 Evaluator 全体リスト

Foundryは 6個カテゴリ, 32個の Evaluatorを提供します:

| カテゴリ | Evaluator 例 |
|---------|---------------|
| **一般品質** | CoherenceEvaluator, FluencyEvaluator, QAEvaluator |
| **テキスト類似も** | SimilarityEvaluator, F1ScoreEvaluator, BleuScoreEvaluator |
| **RAG** | GroundednessEvaluator, RelevanceEvaluator, RetrievalEvaluator |
| **エージェント** | IntentResolutionEvaluator, TaskAdherenceEvaluator, ToolCallAccuracyEvaluator |
| **リスク/安全** | ViolenceEvaluator, SexualEvaluator, ContentSafetyEvaluator |
| **Azure OpenAI Graders** | AzureOpenAILabelGrader, AzureOpenAIGrader |

詳細な内容は [Azure AI Evaluation ドキュメント](https://learn.microsoft.com/en-us/azure/ai-foundry/concepts/evaluation-evaluators)を参照してください.

## 評価基準がして

Foundryは次の 4がないコア評価基準を提供します:

| 基準 | 説明 |
|------|------|
| **Groundedness** | レスポンスが提供されたコンテキスト/知識にベースするはない |
| **Relevance** | レスポンスが質問と関連があるか |
| **Coherence** | レスポンスが論理的で一貫性あるか |
| **Fluency** | レスポンスが自然で文法的で正しいか |

# 簡単なエージェント評価例
from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential

credential = DefaultAzureCredential()
project_client = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=credential)

# テストデータの準備
test_queries = [
  "Pythonでリストをソートする方法は?",
  "機械学習とディープラーニングの違いを説明してください.",
  "ソウルの人口は何人ですか?",
  "最近のAI技術の動向を教えてください.",
  "クラウドコンピューティングのメリットは何ですか?"
]

print("📊 エージェントテスト開始...\n")
print("=" * 80)

# エージェントレスポンス収集
agent_id = "<your-agent-id>" # ⚠️ 03-agentsで作成したエージェント ID

responses = []
for i, query in enumerate(test_queries, 1):
  print(f"\n[質問 {i}/{len(test_queries)}]:{query}")
  
  # Thread 作成
  thread = project_client.agents.create_thread()
  
  # メッセージ送信
  message = project_client.agents.create_message(
    thread_id=thread.id,
    role="user",
    content=query
  )
  
  # Run 実行
  run = project_client.agents.create_and_process_run(
    thread_id=thread.id,
    assistant_id=agent_id
  )
  
  # レスポンス収集
  messages = project_client.agents.list_messages(thread_id=thread.id)
  response = messages.data[0].content[0].text.value
  
  responses.append({"query":query, "response":response})
  print(f"[レスポンス]:{response[:100]}...")
  
print("\n" + "=" * 80)
print(f"\n✅ {len(test_queries)}個の質問テスト完了!")
print(f"\n💡 ポータルでより詳細な評価を行ってください:")
print("  https://ai.azure.com > Build > Evaluations")

### 各基準が重要な有

| Groundedness | Relevance | Coherence | Fluency |
|--------------|-----------|-----------|----------|
| ユーザー 信頼確保 | ユーザー だけ足も向上 | が理解する簡単な回答 | ユーザー 経験向上 |
| 法的責任最小化 | 効率的情報伝達 | 専門的なが未知 | ブランドが未知維持 |
| 虚偽情報防止 | 会話フロー 維持 | 信頼性向上 | がしても増が |

## 評価実行および結果分析

### 評価実行

1. **Submit** ボタンクリック
2. 評価がバックグラウンドで実行なります (約 10-15分)
3. Evaluations ページで進行状況確認

### 結果解釈

### 大規模評価は Portal 使用推奨

**SDK vs Portal 比較:**

| 機能 | Python SDK | Azure Portal |
|------|-----------|--------------|
| サンプルできる | 小規模 (10-20) | 大規模 (50-200+) |
| 可視化 | 制限的 | 豊富なダッシュボード |
| Synthetic データ | 手動作成 | 自動作成 |




















| **Human Evaluation** | 自動評価と並行して新しいで運問題パターン発見 || **ベースラインスコア** | Groundedness ≥4.0 / 残り ≥3.5 / Pass rate ≥80% || **評価周期** | 開発中:毎更新 / デプロイ前:必須 / デプロイ後:週間/月間 || **テストシナリオ** | 一般·複雑·曖昧·多言語質問 + Edge cases || **サンプルできる** | 開発:10-20個 / テスト:50-100個 / 本番環境:200+個 ||------|----------|| 項目 | 推奨事項 |### 推奨事項6. Submit → 結果確認 (10-15分所要)5. Criteria:Groundedness, Relevance, Coherence, Fluency 選択4. Data:Synthetic generation (50-200 サンプル)3. Target:Agent 選択 (はい:ModelRouterAgent)2. + Create new evaluation クリック1. https://ai.azure.com → Build → Evaluations**Portalで評価する方法:**| 協業 | 困難 | チーム共有可能 || 結果管理 | でカラム保存 | クラウド保存および共有 |Pass Rate:85% (43/50 samples)
```

**改善が必要な領域:**
- Groundednessが 4.0 がハインサンプルレビュー
- 失敗した 7個サンプル分析
- Instructions 改善または Knowledge Base 補強

## 📚 追加リソース

- [Azure AI Evaluation 概要](https://learn.microsoft.com/en-us/azure/ai-foundry/concepts/observability?view=foundry#what-are-evaluators)
- [Foundry ポータルで評価実行](https://learn.microsoft.com/en-us/azure/ai-foundry/how-to/evaluate-generative-ai-app?view=foundry)
- [エージェント評価](https://learn.microsoft.com/en-us/azure/ai-foundry/concepts/evaluation-evaluators/agent-evaluators?view=foundry)

## 📚 追加リソース

- [Azure AI Evaluation 概要](https://learn.microsoft.com/en-us/azure/ai-foundry/concepts/observability?view=foundry#what-are-evaluators)
- [Foundry ポータルで評価実行](https://learn.microsoft.com/en-us/azure/ai-foundry/how-to/evaluate-generative-ai-app?view=foundry)
- [エージェント評価](https://learn.microsoft.com/en-us/azure/ai-foundry/concepts/evaluation-evaluators/agent-evaluators?view=foundry)